# 03 - Gold: Transacties (categorisatie)

Bouwt `gold.transacties`: silver-transacties verrijkt met categorie/subcategorie.

Aanpak (Firefly III-stijl regel-engine):
1. **Regels als config** (`config/categorisatie_regels.yaml`) -> geladen in `gold.categorisatie_regels` (wordt elke run herladen, dus versiebeheer je regels in git, niet in de database)
2. **Handmatige overrides** in `gold.categorie_overrides` -> blijvende tabel, wordt NOOIT overschreven door een pipeline-run
3. **Matching**: regex op `naam_omschrijving + mededelingen + tegenrekening`, laagste `prioriteit` wint bij meerdere matches
4. **Voorrang**: handmatige override > regel-match > fallback `Overig/Ongecategoriseerd`
5. **Gold is een TABLE**, herbouwd bij elke pipeline-run (net als silver). Dit is een bewuste snelheidsafweging: de regex-matching tegen alle regels is te duur (~2,5s) om bij elke lees-query (bv. vanuit de API) opnieuw te berekenen. Nadeel: een handmatige override (`zet_override`) is pas zichtbaar in `gold.transacties` na de eerstvolgende `run_gold()`-aanroep, niet direct.

In [ ]:
import sys
from pathlib import Path


def find_project_root(marker="requirements.txt") -> Path:
    p = Path.cwd().resolve()
    for kandidaat in [p, *p.parents]:
        if (kandidaat / marker).exists():
            return kandidaat
    raise RuntimeError("Kon project root niet vinden")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import duckdb
from src.pipeline.paths import DB_PAD
from src.pipeline.transacties import gold

con = duckdb.connect(str(DB_PAD))
print(f"Verbonden met {DB_PAD}")

## 1. Regels, overrides en gold tabel bouwen

Uitgevoerd door `gold.run_gold()` — zie `src/pipeline/transacties/gold.py`:
1. **Regels als config** (`config/categorisatie_regels.yaml`) -> geladen in `gold.categorisatie_regels` (wordt elke run herladen, dus versiebeheer je regels in git, niet in de database)
2. **Handmatige overrides** in `gold.categorie_overrides` -> blijvende tabel, wordt NOOIT overschreven door een pipeline-run
3. **Matching**: regex op `naam_omschrijving + mededelingen + tegenrekening`, laagste `prioriteit` wint bij meerdere matches
4. **Voorrang**: handmatige override > regel-match > fallback `Overig/Ongecategoriseerd`
5. **Gold is een TABLE**, herbouwd bij elke `run_gold()`-aanroep — snel te lezen (zoals `silver.transacties`), maar overrides zijn pas zichtbaar na de eerstvolgende run

In [ ]:
resultaat = gold.run_gold(con)
print(resultaat)

## 2. Verificatie

In [ ]:
print("Verdeling over categorieën:")
display(con.execute("""
    SELECT categorie, subcategorie, COUNT(*) AS aantal, SUM(bedrag_eur) AS totaal
    FROM gold.transacties
    GROUP BY categorie, subcategorie
    ORDER BY aantal DESC
""").df())


In [ ]:

print("\nOngecategoriseerde transacties (kandidaten voor nieuwe regels):")
display(con.execute("""
    SELECT naam_omschrijving, SUM(bedrag_eur)
    FROM gold.transacties
    WHERE categorie = 'Overig'
    GROUP BY naam_omschrijving
    ORDER BY SUM(bedrag_eur) DESC
    LIMIT 100
""").df())


## 3. Handmatig een override toevoegen

Gebruik dit als een regel een transactie verkeerd (of niet) categoriseert. De override wint
altijd van de regel-engine, en blijft bestaan bij toekomstige pipeline-runs.

`zet_override()` staat in `gold.py` en is al geïmporteerd via `from src.pipeline.transacties import gold`.
Zoek eerst het `transactie_id` op via een `SELECT` op `gold.transacties`, en vul die hieronder in.

**Let op**: sinds `gold.transacties` een TABLE is (niet meer een VIEW), moet je na het zetten van een override
`gold.run_gold(con)` opnieuw aanroepen voordat de wijziging zichtbaar wordt in `gold.transacties`.

In [ ]:
# Voorbeeld (uncomment en vul in):
# gold.zet_override(con, "abc123...", "Boodschappen", "Markt", "Weekmarkt, niet in regels")

In [ ]:
con.execute("SELECT * FROM gold.transacties ORDER BY datum DESC LIMIT 10").df()

In [ ]:
con.execute("""
    SELECT naam_omschrijving AS winkel, SUM(bedrag_eur) AS bedrag
    FROM gold.transacties
    WHERE subcategorie = 'Supermarkt'
    GROUP BY naam_omschrijving
    ORDER BY bedrag
    LIMIT 10
""").df()

In [ ]:
con.execute("""
    SELECT YEAR(datum) AS jaar, MONTH(datum) AS maand, SUM(bedrag_eur) AS bedrag
    FROM gold.transacties
    WHERE subcategorie = 'Supermarkt'
    GROUP BY jaar, maand
    ORDER BY jaar DESC, maand DESC
    LIMIT 10
""").df()

In [ ]:
# TODO
# oplossing om queries te draaien en een grafiek weer te geven? in frontend? Misschien queries vastleggen in repo?

In [ ]:
con.close()
